In [14]:
import Pkg

Pkg.activate(".")
Pkg.instantiate()
import Pkg; Pkg.add("KernelAbstractions")

  Activating project at `c:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\repo-cleanup\ocean-modeling\amazon_river\debugging\4\4.3`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\repo-cleanup\ocean-modeling\amazon_river\debugging\4\4.3\Project.toml`
    Manifest No packages added to or removed from `C:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\repo-cleanup\ocean-modeling\amazon_river\debugging\4\4.3\Manifest.toml`


In [15]:
using NumericalEarth
using Oceananigans
using Oceananigans.Units
using Oceananigans.Grids: node
using Oceananigans.TurbulenceClosures: IsopycnalSkewSymmetricDiffusivity, AdvectiveFormulation, VerticalScalarDiffusivity, VerticallyImplicitTimeDiscretization
using Oceananigans.Architectures: architecture
using Oceananigans.Grids: inactive_node
using Oceananigans.Operators: Azá¶œá¶œáµƒ
using Oceananigans.Utils: launch!
using KernelAbstractions: @kernel, @index
using Dates
using Printf
using Statistics
using CUDA

In [16]:
# determining run version/params - :dye_test or :spinup
run_mode = :dye_test

use_rivers = true
resolution_tag = "trial4_2_river_3x3_kz0p1_30m_0p1deg"

# trial 4.2: keep the 3 by 3 river spreading and mix the upper 30 meters.

if run_mode == :dye_test
    run_days = 12
elseif run_mode == :spinup
    run_days = 360
else
    error("need right mode")
end

if use_rivers
    run_name = "rivers_on"
else
    run_name = "rivers_off"
end

if run_mode == :dye_test
    output_tag = "$(run_name)_dye_test_$(resolution_tag)"
else
    output_tag = "$(run_name)_spinup"
end

surface_filename = "amazon_$(output_tag)_surface_fields"
free_surface_filename = "amazon_$(output_tag)_free_surface"
dye_3d_filename = "amazon_$(output_tag)_dye_3d"
salinity_3d_filename = "amazon_$(output_tag)_salinity_3d"

checkpoint_filename = "amazon_$(run_name)_spinup_checkpoint.jld2"

"amazon_rivers_on_spinup_checkpoint.jld2"

In [17]:

arch = GPU()

# grid definitions -- amazon river mouth / plume region
long_west = -58.5
long_east = -41.5
lat_south = -7.8
lat_north = 9.2

# trial 4.2 retains the baseline 0.1-degree horizontal resolution.
# river spreading and local vertical mixing are both turned on.
Nx = 170
Ny = 170
Nz = 20

long_river = -49.5
lat_river  = 0.16

depth = 4000meters # decreased depth to 4k for this test 

z = ExponentialDiscretization(Nz, -depth, 0; scale = depth/4, mutable = false)
underlying_grid = LatitudeLongitudeGrid(
    arch;
    size = (Nx, Ny, Nz),
    halo = (5, 5, 4),
    longitude = (long_west, long_east),
    latitude = (lat_south, lat_north),
    z,
    topology = (Bounded, Bounded, Bounded)
)

bottom_height = regrid_bathymetry(underlying_grid;
                                  minimum_depth = 10,
                                  interpolation_passes = 10, 
                                  major_basins = 1) # only one major basin here -- atlantic 

grid = ImmersedBoundaryGrid(underlying_grid, GridFittedBottom(bottom_height);
                            active_cells_map=true)


                        

[ Info: Loading cached bathymetry from C:\Users\meghn\.julia\scratchspaces\904d977b-046a-4731-8b86-9235c0d1ef02\bathymetry_cache\bathymetry_170x170_-58.5_-41.5_-7.799999999999999_9.2_a0d22c71.jld2


170Ã—170Ã—20 ImmersedBoundaryGrid{Float64, Bounded, Bounded, Bounded} on CUDAGPU with 5Ã—5Ã—4 halo:
â”œâ”€â”€ immersed_boundary: GridFittedBottom(mean(z)=-1077.25, min(z)=-4000.0, max(z)=0.0)
â”œâ”€â”€ underlying_grid: 170Ã—170Ã—20 LatitudeLongitudeGrid{Float64, Bounded, Bounded, Bounded} on CUDAGPU with 5Ã—5Ã—4 halo
â”œâ”€â”€ longitude: Bounded  Î» âˆˆ [-58.5, -41.5] regularly spaced with Î”Î»=0.1
â”œâ”€â”€ latitude:  Bounded  Ï† âˆˆ [-7.8, 9.2]    regularly spaced with Î”Ï†=0.1
â””â”€â”€ z:         Bounded  z âˆˆ [-4000.0, 0.0] variably spaced with min(Î”z)=16.5232, max(Î”z)=738.605

In [18]:

# trial 4.2 keeps the original ordinary gm/redi closure.
eddy_closure = IsopycnalSkewSymmetricDiffusivity(
    Îº_skew = 1e3,
    Îº_symmetric = 1e3,
    skew_flux_formulation = AdvectiveFormulation()
)

vertical_mixing = NumericalEarth.Oceans.default_ocean_closure()

# add extra vertical tracer mixing near the river mouth and above 30 meters
const river_mixing_longitude = -49.5
const river_mixing_latitude = 0.16
const river_mixing_kz = 0.1
const river_mixing_depth = 30meters
const river_mixing_half_width = 0.15

@inline function river_mouth_kz(longitude, latitude, z, time)
    inside_longitude = abs(longitude - river_mixing_longitude) <= river_mixing_half_width
    inside_latitude = abs(latitude - river_mixing_latitude) <= river_mixing_half_width
    inside_depth = z >= -river_mixing_depth

    if inside_longitude && inside_latitude && inside_depth
        return river_mixing_kz
    else
        return 0.0
    end
end

river_mixing = VerticalScalarDiffusivity(
    VerticallyImplicitTimeDiscretization();
    Îº = (T = river_mouth_kz,
         S = river_mouth_kz,
         dye = river_mouth_kz,
         e = 0.0)
)

closure = (eddy_closure, vertical_mixing, river_mixing)

(IsopycnalSkewSymmetricDiffusivity:
â”œâ”€â”€ Îº_skew: 1000.0
â”œâ”€â”€ Îº_symmetric: 1000.0
â”œâ”€â”€ isopycnal_tensor: Oceananigans.TurbulenceClosures.SmallSlopeIsopycnalTensor{Float64}
â””â”€â”€ slope_limiter: Oceananigans.TurbulenceClosures.FluxTapering{Float64}, CATKEVerticalDiffusivity{VerticallyImplicitTimeDiscretization}
â”œâ”€â”€ maximum_tracer_diffusivity: Inf
â”œâ”€â”€ maximum_tke_diffusivity: Inf
â”œâ”€â”€ maximum_viscosity: Inf
â”œâ”€â”€ minimum_tke: 1.0e-9
â”œâ”€â”€ negative_tke_time_scale: 60.0
â”œâ”€â”€ minimum_convective_buoyancy_flux: 1.0e-11
â”œâ”€â”€ tke_time_step: Nothing
â”œâ”€â”€ mixing_length: TKEBasedVerticalDiffusivities.CATKEMixingLength
â”‚   â”œâ”€â”€ CË¢:   1.131
â”‚   â”œâ”€â”€ Cáµ‡:   0.01
â”‚   â”œâ”€â”€ CÊ°â±u: 0.242
â”‚   â”œâ”€â”€ CÊ°â±c: 0.098
â”‚   â”œâ”€â”€ CÊ°â±e: 0.548
â”‚   â”œâ”€â”€ CË¡áµ’u: 0.361
â”‚   â”œâ”€â”€ CË¡áµ’c: 0.369
â”‚   â”œâ”€â”€ CË¡áµ’e: 7.863
â”‚   â”œâ”€â”€ Cáµ˜â¿u: 0.37
â”‚   â”œâ”€â”€ Cáµ˜â¿c: 0.572
â”‚   â”œâ”€â”€ Cáµ˜

In [19]:
free_surface = SplitExplicitFreeSurface(grid; substeps = 70)
momentum_advection = WENOVectorInvariant(order = 5)
tracer_advection = WENO(order = 5)

# create zero-gradient (Neumann) boundary conditions for dye 
# "flow out" part of the model 
dye_bcs = FieldBoundaryConditions(
    west   = GradientBoundaryCondition(0),
    east   = GradientBoundaryCondition(0),
    south  = GradientBoundaryCondition(0),
    north  = GradientBoundaryCondition(0),
    top    = FluxBoundaryCondition(0),
    bottom = FluxBoundaryCondition(0)
)

# compile tracer boundary conditions for the model
model_bcs = (
    dye = dye_bcs,
)

# updated sponge layer logic!
# summary: inital sponge was too large + errored. kept on throwing invalidIREerror. Oceanangians converts the sponge relaxation into a continuous form, which isn't compatible with the GPU, hence the error
# made a manual function creating the gaussian mask for the northern bounday, but then applied the relaxation in discrete form 

# store as gpu-accessible constant
const north_sponge_mask = GaussianMask{:y}(
    center = 9.2,
    width = 0.25
)

# store as a gpu-accessible constant
const north_sponge_rate = 1 / 5days

# apply gaussian dye relaxation in discrete form
@inline function north_dye_sponge(i, j, k, grid, clock, model_fields)
    # get the coordinates of dye cell
    x, y, z = node(i, j, k, grid, Center(), Center(), Center())

    # evaluate gaussian mask from oceaningans at this cell
    mask = north_sponge_mask(x, y, z)

    # read the local dye concentration + relax towards 0 
    dye = @inbounds model_fields.dye[i, j, k]
    return -north_sponge_rate * mask * dye

end

# use the discrete form to avoid the bug 
dye_sponge = Forcing(
    north_dye_sponge;
    discrete_form = true
)

# apply the sponge only to dye
model_forcing = (
    dye = dye_sponge,
)
                        
# dd dye with the original NumericalEarth momentum, tracer, and closure settings.
ocean = ocean_simulation(grid;
                         momentum_advection, tracer_advection, free_surface,
                         closure = closure,
                         tracers = (:T, :S, :dye),
                         boundary_conditions = model_bcs,
                         forcing = model_forcing
                         )

# instead of floating dye patch, new dye patch centered on the river mouth, fading with distance
# dye is initially deposited only in the upper 10 meters
@inline function dye_initial_condition(x, y, z)
    horizontal_width = 0.5

    horizontal_blob = exp(-((x - long_river)^2 + (y - lat_river)^2) / horizontal_width^2)

    if z > -10
        return horizontal_blob
    else
        return 0
    end
end

# print the model structure so we can check that dye is listed as a tracer and that the boundary conditions were set to 0 gradient
@info "We've built an ocean simulation with model:"
@show ocean.model
@show ocean.model.tracers.dye.boundary_conditions



ocean.model = HydrostaticFreeSurfaceModel{CUDAGPU, ImmersedBoundaryGrid}(time = 0 seconds, iteration = 0)
â”œâ”€â”€ grid: 170Ã—170Ã—20 ImmersedBoundaryGrid{Float64, Bounded, Bounded, Bounded} on CUDAGPU with 5Ã—5Ã—4 halo
â”œâ”€â”€ timestepper: SplitRungeKuttaTimeStepper
â”œâ”€â”€ tracers: (T, S, dye, e)
â”œâ”€â”€ closure: Tuple with 3 closures:
â”‚   â”œâ”€â”€ CATKEVerticalDiffusivity{VerticallyImplicitTimeDiscretization}
â”‚   â”œâ”€â”€ IsopycnalSkewSymmetricDiffusivity(Îº_skew=1000.0, Îº_symmetric=1000.0)
â”‚   â””â”€â”€ VerticalScalarDiffusivity{VerticallyImplicitTimeDiscretization}(Î½=0.0, Îº=(T=river_mouth_kz (generic function with 1 method), S=river_mouth_kz (generic function with 1 method), dye=river_mouth_kz (generic function with 1 method), e=0.0))
â”œâ”€â”€ buoyancy: SeawaterBuoyancy with g=9.80665 and BoussinesqEquationOfState{Float64} with gÌ‚ = NegativeZDirection()
â”œâ”€â”€ free surface: SplitExplicitFreeSurface with gravitational acceleration 9.80665 m sâ»Â²
â”‚   â””â”

[ Info: We've built an ocean simulation with model:


Oceananigans.FieldBoundaryConditions, with boundary conditions
â”œâ”€â”€ west: GradientBoundaryCondition: 0.0
â”œâ”€â”€ east: GradientBoundaryCondition: 0.0
â”œâ”€â”€ south: GradientBoundaryCondition: 0.0
â”œâ”€â”€ north: GradientBoundaryCondition: 0.0
â”œâ”€â”€ bottom: FluxBoundaryCondition: 0.0
â”œâ”€â”€ top: FluxBoundaryCondition: 0.0
â””â”€â”€ immersed: FluxBoundaryCondition: Nothing

In [20]:
# ask for ecco credentials at runtime so they are not saved in the notebook
@assert haskey(ENV, "ECCO_USERNAME") "Set ECCO_USERNAME before running this cell"
@assert haskey(ENV, "ECCO_WEBDAV_PASSWORD") "Set ECCO_WEBDAV_PASSWORD before running this cell"

date = DateTime(1993, 1, 1)

ecco_variables = (:temperature, :salinity)
ecco_set = MetadataSet(ecco_variables; dataset = ECCO4Monthly(), date)

set!(ocean.model, ecco_set)
@show keys(ocean.model.tracers)

keys(ocean.model.tracers) = (:T, :S, :dye, :e)


(:T, :S, :dye, :e)

In [21]:

land = JRA55PrescribedLand(arch)
atmosphere = JRA55PrescribedAtmosphere(arch)
ocean_surface = SurfaceRadiationProperties(albedo = LatitudeDependentAlbedo())
radiation = JRA55PrescribedRadiation(arch; ocean_surface)

640Ã—320Ã—1Ã—2920 PrescribedRadiation on LatitudeLongitudeGrid:
â”œâ”€â”€ times: 2920-element StepRangeLen{Float64, Base.TwicePrecision{Float64}, Base.TwicePrecision{Float64}, Int64}
â”œâ”€â”€ stefan_boltzmann_constant: 5.67037e-8
â””â”€â”€ surface_properties: (:ocean, :sea_ice)

In [22]:
# trial 4.2 keeps the 3 by 3 river spreading from trial 4.1
const interface_computations = NumericalEarth.EarthSystemModels.InterfaceComputations

struct RiverSpread{F, A}
    raw_flux :: F
    surface_area :: A
end

# save each active surface cell's area for the mass calculation
@kernel function save_cell_area!(surface_area, grid)
    i, j = @index(Global, NTuple)
    k = grid.Nz
    active = !inactive_node(i, j, k, grid, Center(), Center(), Center())

    if active
        @inbounds surface_area[i, j, 1] = Azá¶œá¶œáµƒ(i, j, k, grid)
    else
        @inbounds surface_area[i, j, 1] = 0
    end
end

# give each source cell's water equally to its active 3 by 3 neighbors
@kernel function spread_river_water!(spread_flux, raw_flux, surface_area, Nx, Ny)
    i, j = @index(Global, NTuple)
    destination_area = @inbounds surface_area[i, j, 1]
    transported_mass = zero(eltype(spread_flux))

    if destination_area > 0
        for source_i in max(1, i - 1):min(Nx, i + 1)
            for source_j in max(1, j - 1):min(Ny, j + 1)
                source_area = @inbounds surface_area[source_i, source_j, 1]

                if source_area > 0
                    active_neighbors = 0
                    for neighbor_i in max(1, source_i - 1):min(Nx, source_i + 1)
                        for neighbor_j in max(1, source_j - 1):min(Ny, source_j + 1)
                            neighbor_area = @inbounds surface_area[neighbor_i, neighbor_j, 1]
                            if neighbor_area > 0
                                active_neighbors += 1
                            end
                        end
                    end

                    source_flux = @inbounds raw_flux[source_i, source_j, 1]
                    transported_mass += source_flux * source_area / active_neighbors
                end
            end
        end

        @inbounds spread_flux[i, j, 1] = transported_mass / destination_area
    else
        @inbounds spread_flux[i, j, 1] = zero(eltype(spread_flux))
    end
end

function RiverSpread(grid)
    raw_flux = Field{Center, Center, Nothing}(grid)
    surface_area = Field{Center, Center, Nothing}(grid)
    launch!(architecture(grid), grid, :xy, save_cell_area!, surface_area, grid)
    return RiverSpread(raw_flux, surface_area)
end

# save the original flux, then replace it with the spread flux
function interface_computations.correct_state!(correction::RiverSpread, exchanger, grid)
    freshwater_flux = exchanger.state.freshwater_flux
    copyto!(parent(correction.raw_flux.data), parent(freshwater_flux.data))
    launch!(architecture(grid), grid, :xy, spread_river_water!,
            freshwater_flux, correction.raw_flux, correction.surface_area, grid.Nx, grid.Ny)
    return nothing
end

# attach this correction when numericalearth makes the land exchange field
function interface_computations.ComponentExchanger(::PrescribedLand, grid)
    state = (; freshwater_flux = Field{Center, Center, Nothing}(grid))
    correction = RiverSpread(grid)
    return interface_computations.ComponentExchanger(state, nothing, correction)
end

In [23]:
# build the coupled model so numericalearth prepares the river forcing
@assert use_rivers "trial 4.3 requires use_rivers = true"
land_component = land

coupled_model = EarthSystemModel(
    ;
    ocean,
    atmosphere,
    land = land_component,
    radiation
)

# retrieve the original flux, spread flux, and cell areas from the exchanger
land_exchange = coupled_model.interfaces.exchanger.land
river_spreading = land_exchange.correction
raw_flux = Array(interior(river_spreading.raw_flux, :, :, 1))
spread_flux = Array(interior(land_exchange.state.freshwater_flux, :, :, 1))
surface_area = Array(interior(river_spreading.surface_area, :, :, 1))

# confirm that the 3 by 3 spreading conserved total freshwater transport
raw_transport = sum(raw_flux .* surface_area)
spread_transport = sum(spread_flux .* surface_area)
conservation_error = abs(spread_transport - raw_transport) / max(abs(raw_transport), eps())

@info "trial 4.3 freshwater spreading check" raw_transport spread_transport conservation_error maximum(raw_flux) maximum(spread_flux)
@assert conservation_error < 1e-10 "3 by 3 spreading did not conserve freshwater transport"

â”Œ Warning: 1440Ã—720Ã—1Ã—365 PrescribedLand tracks time as Float32 but the EarthSystemModel clock uses Float64; coercing the component clock to keep components synchronized.
â”” @ NumericalEarth.EarthSystemModels C:\Users\meghn\.julia\packages\NumericalEarth\eKhWQ\src\EarthSystemModels\components.jl:136
â”Œ Info: trial 4.3 freshwater spreading check
â”‚   raw_transport = 1.1018805993114366e8
â”‚   spread_transport = 1.1018805993114367e8
â”‚   conservation_error = 1.3523390105206833e-16
â”‚   maximum(raw_flux) = 0.12167064845561981
â””   maximum(spread_flux) = 0.08382366267590947


In [ ]:
# trial 4.3: measure how much amazon freshwater falls inside the mixing box

# physical coordinates of the horizontal cell centers
longitude_centers = Array(Î»nodes(grid, Center()))
latitude_centers = Array(Ï†nodes(grid, Center()))

# keep only positive freshwater input for this diagnostic
positive_spread_flux = max.(spread_flux, 0.0)

# mark the broad amazon region and the smaller trial 4.2 mixing box
amazon_window = falses(size(spread_flux))
mixing_box = falses(size(spread_flux))

for j in eachindex(latitude_centers)
    for i in eachindex(longitude_centers)
        longitude = longitude_centers[i]
        latitude = latitude_centers[j]

        amazon_window[i, j] =
            abs(longitude - long_river) <= 2.0 &&
            abs(latitude - lat_river) <= 2.0

        mixing_box[i, j] =
            abs(longitude - river_mixing_longitude) <= river_mixing_half_width &&
            abs(latitude - river_mixing_latitude) <= river_mixing_half_width
    end
end

# cells receiving positive freshwater in each region
amazon_forced_mask = amazon_window .& (positive_spread_flux .> 0)
mixing_forced_mask = mixing_box .& (positive_spread_flux .> 0)

# flux times area gives freshwater transport in each cell
amazon_transport = sum(positive_spread_flux[amazon_window] .* surface_area[amazon_window])

mixing_transport = sum(positive_spread_flux[mixing_box] .* surface_area[mixing_box])
# main result: fraction of nearby amazon input inside the mixing box
mixing_transport_fraction = mixing_transport / amazon_transport

# useful counts for interpreting the fraction
amazon_forced_cell_count = count(amazon_forced_mask)
mixing_forced_cell_count = count(mixing_forced_mask)
mixing_ocean_cell_count = count(mixing_box .& (surface_area .> 0))

# location of the strongest positive freshwater cell near the amazon
amazon_flux = copy(positive_spread_flux)
amazon_flux[.!amazon_window] .= 0.0
strongest_flux, strongest_cell = findmax(amazon_flux)
strongest_longitude = longitude_centers[strongest_cell[1]]
strongest_latitude = latitude_centers[strongest_cell[2]]
strongest_inside_mixing_box = mixing_box[strongest_cell]

# after trying a few nearby centers and widths, test a larger box centered on
# the measured strongest cell for trial 4.4
candidate_mixing_longitude = strongest_longitude
candidate_mixing_latitude = strongest_latitude
candidate_mixing_half_width = 0.25 # make it larger
candidate_mixing_box = falses(size(spread_flux))

for j in eachindex(latitude_centers)
    for i in eachindex(longitude_centers)
        longitude = longitude_centers[i]
        latitude = latitude_centers[j]

        candidate_mixing_box[i, j] =
            abs(longitude - candidate_mixing_longitude) <= candidate_mixing_half_width &&
            abs(latitude - candidate_mixing_latitude) <= candidate_mixing_half_width
    end
end

# check how much transport the candidate box would cover
candidate_mixing_transport =
    sum(positive_spread_flux[candidate_mixing_box] .* surface_area[candidate_mixing_box])

candidate_mixing_transport_fraction =
    candidate_mixing_transport / amazon_transport

candidate_forced_cell_count =
    count(candidate_mixing_box .& (positive_spread_flux .> 0))

candidate_strongest_inside =
    candidate_mixing_box[strongest_cell]

@info "trial 4.3 original mixing box" mixing_transport mixing_transport_fraction mixing_forced_cell_count mixing_ocean_cell_count strongest_inside_mixing_box

@info "candidate box for trial 4.4" candidate_mixing_longitude candidate_mixing_latitude candidate_mixing_half_width candidate_mixing_transport candidate_mixing_transport_fraction candidate_forced_cell_count candidate_strongest_inside


â”Œ Info: trial 4.3 original mixing box
â”‚   mixing_transport = 1.3940266967376398e7
â”‚   mixing_transport_fraction = 0.13582416731723226
â”‚   mixing_forced_cell_count = 10
â”‚   mixing_ocean_cell_count = 12
â””   strongest_inside_mixing_box = false
â”Œ Info: trial 4.3 candidate box for trial 4.4
â”‚   candidate_mixing_longitude = -49.35
â”‚   candidate_mixing_latitude = -0.15000000000000063
â”‚   candidate_mixing_half_width = 0.25
â”‚   candidate_mixing_transport = 9.062768663539404e7
â”‚   candidate_mixing_transport_fraction = 0.8830125062846002
â”‚   candidate_forced_cell_count = 25
â””   candidate_strongest_inside = true
